# CrashDiag parent-SFT hard baseline on Kaggle

This independent notebook evaluates the signed parent SFT adapter against the exact same signed 192-row schema-v2 hard evaluation split used for `CrashDiag GRPO v1 candidate`. It uses deterministic generation and live mechanical sandbox resolution checks. No LLM grades another LLM. The baseline report and a direct per-fault comparison are uploaded as a separate immutable private-bucket run.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

WORKFLOW_VERSION = "parent-sft-hard-baseline-v1"
REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
SANDBOX_URL = "https://sandbox.devaanshpathak.com"
HARD_RUN_ID = os.environ.get("CRASHDIAG_HARD_RUN_ID") or "20260720T164228Z-grpo-hard-7aa31d7f3710"
HARD_SOURCE_COMMIT = os.environ.get("CRASHDIAG_HARD_SOURCE_COMMIT") or "f732e5fed815a73a53c8ee860c3fd865a6577fb2"
EVALUATOR_COMMIT = os.environ.get("CRASHDIAG_EVALUATOR_COMMIT") or "1df32f6904585768c6e929a43b7eaa96974c0c67"
BASELINE_RUN_ID = os.environ.get("CRASHDIAG_BASELINE_RUN_ID") or "20260720T164228Z-parent-sft-hard-baseline-7aa31d7f3710"
BASELINE_STAGE = "parent-sft-hard-evaluation"
COMPARISON_STAGE = "grpo-v1-comparison"
PRECISION = "auto"
EXPECTED_ROWS = 192

for name, value in (("HARD_SOURCE_COMMIT", HARD_SOURCE_COMMIT), ("EVALUATOR_COMMIT", EVALUATOR_COMMIT)):
    if re.fullmatch(r"[0-9a-f]{40}", value) is None:
        raise ValueError(f"{name} must be a full lowercase Git SHA")
print(f"WORKFLOW_VERSION={WORKFLOW_VERSION}\nHARD_RUN_ID={HARD_RUN_ID}\nBASELINE_RUN_ID={BASELINE_RUN_ID}\nEVALUATOR_COMMIT={EVALUATOR_COMMIT}")

## Install the exact evaluator revision used for the GRPO candidate

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", EVALUATOR_COMMIT], check=True)
CURRENT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if CURRENT_COMMIT != EVALUATOR_COMMIT:
    raise RuntimeError("evaluator checkout mismatch")
torchao_probe = subprocess.run([sys.executable, "-m", "pip", "show", "torchao"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if torchao_probe.returncode == 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"checked_out_evaluator={CURRENT_COMMIT}")

## Load Kaggle secrets without displaying them

In [ ]:
def required_secret(name: str) -> str:
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach Kaggle Secret {name!r}") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_SANDBOX_TOKEN"] = required_secret("CRASHDIAG_SANDBOX_TOKEN")
os.environ["CRASHDIAG_SANDBOX_URL"] = SANDBOX_URL
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = BASELINE_RUN_ID
print("Kaggle Secrets loaded; values were not printed.")

## Download and verify the hard split and its signed parent SFT adapter

In [ ]:
from training.artifacts import ArtifactConfig, ArtifactUploader
from training.calibrate_grpo import read_jsonl
from training.generate_grpo_hard import read_parent_reference

def make_uploader(run_id: str) -> ArtifactUploader:
    return ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=run_id, prefix="runs", policy="required", local_root=REPO_DIR / "artifacts", token=os.environ["HF_TOKEN"]))

def fetch_stage(client: ArtifactUploader, stage: str, target: Path, paths: list[str]) -> Path:
    if target.exists() and any(target.iterdir()):
        client.verify_local_stage(target, stage, include_paths=paths)
    else:
        client.download_stage(stage, target, include_paths=paths)
    return target

def remote_path_exists(client: ArtifactUploader, path: str) -> bool:
    return any(getattr(item, "path", None) == path for item in client.api.get_bucket_paths_info(client.config.bucket_id, [path]))

HANDOFF = Path("/kaggle/working/crashdiag-parent-hard-baseline")
hard_client = make_uploader(HARD_RUN_ID)
HARD_DATA = fetch_stage(hard_client, "datasets", HANDOFF / HARD_RUN_ID / "datasets", ["grpo_hard_eval.jsonl", "grpo_hard_summary.json", "parent_sft.json"])
hard_manifest = json.loads((HARD_DATA / "manifest.json").read_text())
if hard_manifest.get("runtime", {}).get("git_commit") != HARD_SOURCE_COMMIT:
    raise RuntimeError("hard dataset/source commit mismatch")
hard_summary = json.loads((HARD_DATA / "grpo_hard_summary.json").read_text())
if hard_summary.get("curriculum_version") != 2 or hard_summary.get("action_contract") != "parameter_free_repairs":
    raise RuntimeError("hard dataset is not curriculum-v2 parameter-free data")
HARD_EVAL_FILE = HARD_DATA / "grpo_hard_eval.jsonl"
if len(read_jsonl(HARD_EVAL_FILE)) != EXPECTED_ROWS:
    raise RuntimeError("hard evaluation split is not exactly 192 rows")

parent_pointer = json.loads((HARD_DATA / "parent_sft.json").read_text())
PARENT_SFT_RUN_ID = parent_pointer["run_id"]
parent_client = make_uploader(PARENT_SFT_RUN_ID)
PARENT_SFT_DIR = fetch_stage(parent_client, "sft", HANDOFF / PARENT_SFT_RUN_ID / "sft", ["adapter_config.json", "adapter_model.safetensors", "chat_template.jinja", "tokenizer.json", "tokenizer_config.json"])
verified_parent = read_parent_reference(PARENT_SFT_DIR, PARENT_SFT_RUN_ID)
for key in ("manifest_sha256", "adapter_sha256", "base_model"):
    if verified_parent[key] != parent_pointer[key]:
        raise RuntimeError(f"parent SFT handoff mismatch: {key}")
print(f"verified exact {EXPECTED_ROWS}-row hard split and parent SFT run {PARENT_SFT_RUN_ID}")

## Probe the live schema-v2 sandbox and Kaggle GPU

In [ ]:
from crashdiag.sandbox_apps.http import HttpSandbox
import torch

with HttpSandbox(SANDBOX_URL, api_token=os.environ["CRASHDIAG_SANDBOX_TOKEN"], timeout=15.0) as sandbox:
    service = sandbox.service_health()
    application = sandbox.health_check()
if 2 not in service.get("scenario_schema_versions", []):
    raise RuntimeError(f"Vultr service lacks schema v2: {service}")
if service.get("hard_scenario_batch") is not True:
    raise RuntimeError("Vultr service lacks atomic hard-scenario setup")
if application.get("healthy") is not True:
    raise RuntimeError(f"sandbox preflight failed: {application}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU")
print(f"gpu={torch.cuda.get_device_name(0)}, bf16={torch.cuda.is_bf16_supported()}")

## Evaluate or restore the immutable parent-SFT hard baseline

In [ ]:
from training.evaluate_jsonl import main as evaluate_jsonl_main
from IPython.display import SVG, display

baseline_uploader = make_uploader(BASELINE_RUN_ID)
BASELINE_OUTPUT = REPO_DIR / "outputs/parent-sft-hard-evaluation"
BASELINE_CACHE = HANDOFF / BASELINE_RUN_ID / BASELINE_STAGE
BASELINE_PATHS = ["mechanical_evaluation.json", "reports/mechanical_evaluation_summary.json", "reports/mechanical_success_by_fault.svg"]
BASELINE_COMPLETE = baseline_uploader.stage_is_complete(BASELINE_STAGE)
COMPARISON_COMPLETE = baseline_uploader.stage_is_complete(COMPARISON_STAGE)
RUN_SUCCESS_PATH = f"{baseline_uploader.config.remote_root}/_SUCCESS.json"
RUN_COMPLETE = remote_path_exists(baseline_uploader, RUN_SUCCESS_PATH)
RUN_OPENED = False
if RUN_COMPLETE and not (BASELINE_COMPLETE and COMPARISON_COMPLETE):
    raise RuntimeError("baseline run is complete but required stages are missing")
if not RUN_COMPLETE and not (BASELINE_COMPLETE and COMPARISON_COMPLETE):
    baseline_uploader.start_run({"workflow": WORKFLOW_VERSION, "hard_run_id": HARD_RUN_ID, "parent_sft_run_id": PARENT_SFT_RUN_ID, "evaluator_commit": EVALUATOR_COMMIT, "scoring": "mechanical_fault_resolution"})
    RUN_OPENED = True

if BASELINE_COMPLETE:
    BASELINE_DIR = fetch_stage(baseline_uploader, BASELINE_STAGE, BASELINE_CACHE, BASELINE_PATHS)
    print(f"reused signed stage {BASELINE_STAGE}")
else:
    BASELINE_DIR = BASELINE_OUTPUT
    baseline_exit = evaluate_jsonl_main(["--model", str(PARENT_SFT_DIR), "--dataset", str(HARD_EVAL_FILE), "--output-dir", str(BASELINE_DIR), "--precision", PRECISION, "--artifact-stage", BASELINE_STAGE])
    if baseline_exit != 0:
        raise RuntimeError(f"parent SFT evaluation failed with status {baseline_exit}")

baseline_summary = json.loads((BASELINE_DIR / "reports/mechanical_evaluation_summary.json").read_text())
if sum(int(value["episodes"]) for value in baseline_summary["per_fault"].values()) != EXPECTED_ROWS:
    raise RuntimeError("baseline report did not evaluate all 192 hard rows")
baseline_report = json.loads((BASELINE_DIR / "mechanical_evaluation.json").read_text())
if baseline_report["summary"].get("backend_error_rate") != 0.0:
    raise RuntimeError("baseline evaluation had sandbox backend errors")
print(json.dumps(baseline_summary, indent=2, sort_keys=True))
for chart in sorted((BASELINE_DIR / "reports").glob("*.svg")):
    display(SVG(filename=str(chart)))

## Compare parent SFT with GRPO v1 and sign the baseline run

In [ ]:
GRPO_EVAL = fetch_stage(hard_client, "hard-evaluation", HANDOFF / HARD_RUN_ID / "hard-evaluation", ["reports/mechanical_evaluation_summary.json"])
grpo_summary = json.loads((GRPO_EVAL / "reports/mechanical_evaluation_summary.json").read_text())
faults = sorted(grpo_summary["per_fault"])
comparison = {
    "schema_version": 1,
    "scoring": "mechanical_fault_resolution",
    "hard_run_id": HARD_RUN_ID,
    "parent_sft_run_id": PARENT_SFT_RUN_ID,
    "rows": EXPECTED_ROWS,
    "parent_sft_success_rate": baseline_summary["overall_success_rate"],
    "grpo_v1_success_rate": grpo_summary["overall_success_rate"],
    "absolute_delta": grpo_summary["overall_success_rate"] - baseline_summary["overall_success_rate"],
    "per_fault": {
        fault: {
            "parent_sft": baseline_summary["per_fault"][fault]["success_rate"],
            "grpo_v1": grpo_summary["per_fault"][fault]["success_rate"],
            "absolute_delta": grpo_summary["per_fault"][fault]["success_rate"] - baseline_summary["per_fault"][fault]["success_rate"],
        }
        for fault in faults
    },
}
COMPARISON_CACHE = HANDOFF / BASELINE_RUN_ID / COMPARISON_STAGE
if COMPARISON_COMPLETE:
    COMPARISON_DIR = fetch_stage(baseline_uploader, COMPARISON_STAGE, COMPARISON_CACHE, ["comparison.json", "comparison.md"])
    comparison = json.loads((COMPARISON_DIR / "comparison.json").read_text())
    print(f"reused signed stage {COMPARISON_STAGE}")
else:
    COMPARISON_DIR = REPO_DIR / "outputs/parent-sft-vs-grpo-v1"
    COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    (COMPARISON_DIR / "comparison.json").write_text(json.dumps(comparison, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
    lines = ["# Parent SFT versus CrashDiag GRPO v1", "", f"Exact hard rows: {EXPECTED_ROWS}", "", "| Fault | Parent SFT | GRPO v1 | Delta |", "| --- | ---: | ---: | ---: |"]
    for fault in faults:
        values = comparison["per_fault"][fault]
        lines.append(f"| {fault} | {values['parent_sft']:.2%} | {values['grpo_v1']:.2%} | {values['absolute_delta']:+.2%} |")
    lines.extend(["", f"Overall: {comparison['parent_sft_success_rate']:.2%} -> {comparison['grpo_v1_success_rate']:.2%} ({comparison['absolute_delta']:+.2%}).", "", "All success values are mechanical sandbox-resolution checks; no LLM grading was used."])
    (COMPARISON_DIR / "comparison.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    baseline_uploader.start_stage(COMPARISON_STAGE, {"hard_run_id": HARD_RUN_ID, "parent_sft_run_id": PARENT_SFT_RUN_ID, "rows": EXPECTED_ROWS, "scoring": "mechanical_fault_resolution"})
    baseline_uploader.upload_files([COMPARISON_DIR / "comparison.json", COMPARISON_DIR / "comparison.md"], COMPARISON_STAGE, metadata=comparison)

if not RUN_COMPLETE:
    baseline_uploader.complete_run({"stages": [BASELINE_STAGE, COMPARISON_STAGE], "workflow": WORKFLOW_VERSION, "hard_run_id": HARD_RUN_ID, "parent_sft_run_id": PARENT_SFT_RUN_ID, "evaluator_commit": EVALUATOR_COMMIT, "scoring": "mechanical_fault_resolution"})
print(json.dumps(comparison, indent=2, sort_keys=True))
print(f"BASELINE_ARTIFACTS={baseline_uploader.remote_uri()}")